# ESIS Mobile V2 â€” Kaggle APK Build

**Edge Survival Intelligence System** â€” Gemma 4 powered crisis intervention for people experiencing homelessness.

## What this builds
- Gemma 4 (HuggingFace) powered intervention plans
- Housing track assignment across 9 program tracks
- Advocacy packet generation (one-page summary, advocate script, referral handoff)
- Police interaction log
- Community ping (real-time crisis location sharing)

## Instructions
1. **Run All** â€” build takes ~10â€“15 minutes
2. Download **`esis-v2-debug.apk`** from the Output panel (right side)

Produces a debug APK installable on any Android phone (enable *Install unknown apps* in settings).

In [ ]:
%%bash
# Cell 1 â€” Install Node 20 and Java 17
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - 2>/dev/null
sudo apt-get install -y nodejs 2>/dev/null | tail -3

node --version
npm --version

sudo apt-get install -y openjdk-17-jdk 2>/dev/null | tail -3
java -version

In [ ]:
%%bash
# Cell 2 â€” Install Android SDK command-line tools
cd /tmp

wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip \
  -O cmdtools.zip

# Unzip to temp dir, then move into the required layout
unzip -q cmdtools.zip -d cmdtools_extract
mkdir -p android-sdk/cmdline-tools/latest
mv cmdtools_extract/cmdline-tools/* android-sdk/cmdline-tools/latest/

export ANDROID_HOME=/tmp/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools

yes | sdkmanager --no_https --licenses 2>/dev/null | tail -1
sdkmanager --no_https "platform-tools" "build-tools;34.0.0" "platforms;android-34" 2>&1 | tail -5
echo "Android SDK ready."

In [ ]:
%%bash
# Cell 3 â€” Clone repo and install npm deps
rm -rf /tmp/esis-repo /tmp/esis-mobile

git clone --depth 1 https://github.com/trextrader/esis.git /tmp/esis-repo
cp -r /tmp/esis-repo/mobile /tmp/esis-mobile
cd /tmp/esis-mobile

npm install --legacy-peer-deps 2>&1 | tail -8
echo "npm install done"

In [ ]:
%%bash
# Cell 4 â€” TypeScript check (catches type errors before the slow Gradle build)
cd /tmp/esis-mobile

npx tsc --noEmit 2>&1
echo "TypeScript check complete (exit $?)"

In [ ]:
%%bash
# Cell 5 â€” Prebuild Android project with Expo
export ANDROID_HOME=/tmp/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools

# Disable Expo's network calls â€” Kaggle blocks expo.dev SSL
export EXPO_NO_TELEMETRY=1
export EXPO_SKIP_CLI_UPGRADE_CHECK=1
export CI=1

cd /tmp/esis-mobile

npx expo prebuild --platform android --clean 2>&1 | tail -15
echo "Prebuild complete"

In [ ]:
%%bash
# Cell 6 — Patch signing config and build release APK
export ANDROID_HOME=/tmp/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64

cd /tmp/esis-mobile/android

# Patch app/build.gradle: add signingConfig signingConfigs.debug to the release
# buildType so assembleRelease produces an installable APK without a separate keystore.
python3 << 'PYEOF2'
import re

with open('app/build.gradle') as f:
    txt = f.read()

# Only patch if not already present
if 'signingConfig signingConfigs.debug' not in txt.split('release {', 1)[-1][:300]:
    txt = txt.replace('release {', 'release {
            signingConfig signingConfigs.debug', 1)
    with open('app/build.gradle', 'w') as f:
        f.write(txt)
    print('build.gradle patched: release will sign with debug keystore')
else:
    print('build.gradle already configured')
PYEOF2

chmod +x gradlew
./gradlew assembleRelease --no-daemon 2>&1 | tail -25
echo "Build complete"


In [ ]:
import shutil, os

src = '/tmp/esis-mobile/android/app/build/outputs/apk/release/app-release.apk'
dst = '/kaggle/working/esis-v2.apk'

shutil.copy(src, dst)
size_mb = os.path.getsize(dst) / 1024 / 1024
print(f'APK ready: {dst} ({size_mb:.1f} MB)')
print('Download from the Output panel on the right →')
print()
print('Install on Android: Settings → Apps → Install unknown apps → enable for your browser/Files app')
